In [1]:
from pathlib import Path
import sys, json
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "Module_01_User_and_Body_Profile.md").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from IPython.display import display, Markdown


# 04 — Normalization, validation & review

**Research question:** Dữ liệu OCR có giữ nguồn gốc và cần xác nhận trước tạo observation không?

Baseline: schema versioned + kiểm tra xác định, độc lập LLM. Thí nghiệm dùng ví dụ giả lập lỗi thập phân và confidence thấp; không dùng người/phiếu thật.

In [2]:
from src.document_ai.parser import parse_lines
from src.document_ai.schema import extraction_schema
from jsonschema import Draft202012Validator, FormatChecker
draft = parse_lines([
    {"text": "Weight: 700 kg", "confidence": 0.72},
    {"text": "PBF: 25,5 %", "confidence": 0.99},
    {"text": "Test date: 2026-10-21", "confidence": 0.99},
], document_id="synthetic-review-demo", model_version="synthetic-error-fixture")
Draft202012Validator(extraction_schema(), format_checker=FormatChecker()).validate(draft)
print(draft["verification_status"])
print(draft["quality_flags"])

pending_user_review
['basal_metabolic_rate_kcal:missing', 'bmi:missing', 'body_fat_mass_kg:missing', 'ecw_tbw_ratio:missing', 'height_cm:missing', 'skeletal_muscle_mass_kg:missing', 'total_body_water_l:missing', 'weight_kg:low_confidence', 'weight_kg:outside_plausibility_bounds']


In [3]:
from src.document_ai.review import confirm_fields
try:
    confirm_fields(draft, reviewer_id="synthetic-reviewer", selected_fields=["weight_kg"], processing_consent=True)
except ValueError as exc:
    print("Blocked until reviewed:", exc)
# Demo of an explicit human correction; a future API must supply real auth/consent.
review = confirm_fields(draft, reviewer_id="synthetic-reviewer", selected_fields=["weight_kg", "percent_body_fat"],
                        corrections={"weight_kg": 70.0}, processing_consent=True)
display(review)
assert draft["weight_kg"] == 700 and draft["verification_status"] == "pending_user_review"
assert review["training_consent"] is False
assert all(o["provenance"] == "extracted" for o in review["observations"])
print("Original draft retained; separate confirmed observations created.")

Blocked until reviewed: Review and acknowledge flags first: ['weight_kg:low_confidence', 'weight_kg:outside_plausibility_bounds']


{'review_id': '261f7c7c-98b6-46ab-b9db-407d951c87af',
 'document_id': 'synthetic-review-demo',
 'reviewer_id': 'synthetic-reviewer',
 'reviewed_at': '2026-10-21T16:35:08.721020+00:00',
 'extraction_sha256': '667edbddd34ce3b8757a5ab54ae3b99f846cdbc42bded23bc827228502de7270',
 'processing_consent': True,
 'training_consent': False,
 'observations': [{'field': 'weight_kg',
   'value': 70.0,
   'unit': 'kg',
   'source': 'inbody',
   'provenance': 'extracted',
   'measurement_method': 'device_report',
   'verification_status': 'confirmed_by_user',
   'measured_at': '2026-10-21',
   'quality_flags': [],
   'recorded_at': '2026-10-21T16:35:08.721020+00:00',
   'model_version': 'synthetic-error-fixture'},
  {'field': 'percent_body_fat',
   'value': 25.5,
   'unit': '%',
   'source': 'inbody',
   'provenance': 'extracted',
   'measurement_method': 'device_report',
   'verification_status': 'confirmed_by_user',
   'measured_at': '2026-10-21',
   'quality_flags': [],
   'recorded_at': '2026-10-2

Original draft retained; separate confirmed observations created.


## Metrics / error analysis / conclusion

Unit tests bao phủ mất đơn vị, nhãn trùng, outlier, cross-field inconsistency, confidence, consent và audit. Sửa số tạo record riêng; không biến chỉ số BIA thành ground truth, không ghi đè dữ liệu đo đã có. Chưa có UX review, DB/auth hay consent store production. Bước tiếp theo là tài liệu thật được cấp quyền và kiểm tra annotation độc lập trước tích hợp API.